In [12]:
import pandas as pd
from transformers import BertTokenizer, BertForMaskedLM
import torch
from tqdm import tqdm
import numpy as np

# 加载预训练的BERT模型和分词器
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
model = BertForMaskedLM.from_pretrained('bert-base-chinese')

# 读取数据集
df = pd.read_csv('selected_dev_dataset.csv')

results = []

# 使用tqdm创建进度条
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing rows"):
    sentence = row['tagged']  # 使用'tagged'列作为句子
    correct_word = row['classifier']  # 使用'classifier'列作为正确答案

    # 替换<CL>为[MASK]，替换<h>和</h>为空字符串
    sentence = sentence.replace('<CL>', '[MASK]').replace('<h>', '').replace('</h>', '')

    # 分词
    inputs = tokenizer(sentence, return_tensors='pt')

    # 获取预测结果
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # 原始logits（未归一化的分数）

    # 获取[MASK]位置的预测结果
    mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1]

    if mask_token_index.numel() == 0:
        print(f"\nWarning: No [MASK] token found in sentence: {sentence}")
        # 如果没有找到[MASK]，记录特殊的值
        top1_word = "N/A"
        top1_logprob = "N/A"
        top3_words = ["N/A", "N/A", "N/A"]
        top3_logprobs = ["N/A", "N/A", "N/A"]
    else:
        # 获取[MASK]位置的logits
        mask_logits = logits[0, mask_token_index[0]]

        # 计算softmax概率
        probs = torch.nn.functional.softmax(mask_logits, dim=-1)

        # 计算对数概率（log probabilities）
        log_probs = torch.log(probs)

        # 获取topk预测结果 (k=3)
        topk_logprobs, topk_token_ids = torch.topk(log_probs, k=3)
        topk_logprobs = topk_logprobs.tolist()
        topk_token_ids = topk_token_ids.tolist()

        # 解码token
        topk_words = tokenizer.convert_ids_to_tokens(topk_token_ids)

        # 获取top1结果
        top1_word = topk_words[0]
        top1_logprob = topk_logprobs[0]

        # 获取top3结果
        top3_words = topk_words[:3]
        top3_logprobs = topk_logprobs[:3]

    # 将结果保存到列表中
    results.append([
        row['new_index'],
        sentence,
        correct_word,
        top1_word,
        top1_logprob,
        ", ".join(top3_words),
        ", ".join([f"{lp:.4f}" for lp in top3_logprobs])
    ])

    # 打印当前处理行的信息
    print(f"\nProcessed row {index + 1}/{len(df)}")
    print(f"Sentence: {sentence}")
    print(f"Target: {correct_word}")
    print(f"Top1 prediction: {top1_word} (logprob: {top1_logprob:.4f})")
    print(f"Top3 predictions: {', '.join(top3_words)}")
    print(f"Top3 log probabilities: {', '.join([f'{lp:.4f}' for lp in top3_logprobs])}")

# 将结果保存到新的CSV文件
result_df = pd.DataFrame(results, columns=[
    'new_index',
    'sentence',
    'target',
    'top1_prediction',
    'top1_log_probability',
    'top3_predictions',
    'top3_log_probabilities'
])
result_df.to_csv('bert_mlm_result.csv', index=False, encoding='utf-8-sig')

print("\nResults saved to bert_mlm_result.csv")

Some weights of the model checkpoint at bert-base-chinese were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Processing rows:   0%|          | 3/1937 [00:00<02:11, 14.70it/s]


Processed row 1/1937
Sentence: 无意 间 转到 一 [MASK] 摩托车 网站 , 看 着 这些 曾经 情迷 到 疯狂 的 机器 。
Target: 个
Top1 prediction: 个 (logprob: -0.2823)
Top3 predictions: 个, 些, 家
Top3 log probabilities: -0.2823, -2.1352, -2.2199

Processed row 2/1937
Sentence: 上传 了 1 [MASK] 照片 到 相册 真实 的 自己
Target: 张
Top1 prediction: 张 (logprob: -0.0256)
Top3 predictions: 张, 个, 幅
Top3 log probabilities: -0.0256, -5.2300, -5.3693

Processed row 3/1937
Sentence: 上传 了 1 [MASK] 照片 到 相册 2012 新 气象
Target: 张
Top1 prediction: 张 (logprob: -0.0097)
Top3 predictions: 张, 个, 幅
Top3 log probabilities: -0.0097, -6.0123, -6.0582

Processed row 4/1937
Sentence: 数 [MASK] 盘石 上 , 几 叶 落 云 中 。
Target: 枝
Top1 prediction: 枝 (logprob: -1.7365)
Top3 predictions: 枝, 人, 峰
Top3 log probabilities: -1.7365, -2.8033, -2.9454


Processing rows:   0%|          | 7/1937 [00:00<02:28, 13.00it/s]


Processed row 5/1937
Sentence: 哎 , 把 一 [MASK] 电脑 修 的 加电 无 显 了 , 悲催 的 新年 开始 。
Target: 台
Top1 prediction: 台 (logprob: -0.2036)
Top3 predictions: 台, 个, 部
Top3 log probabilities: -0.2036, -2.5749, -3.6483

Processed row 6/1937
Sentence: 集团 职能 片 年会 , 多 喝 了 两 [MASK] 绿 茶 , 结果 到 现在 还 很 有精神 , 悲催 啊 。
Target: 杯
Top1 prediction: 杯 (logprob: -0.5503)
Top3 predictions: 杯, 瓶, 壶
Top3 log probabilities: -0.5503, -2.3104, -2.8615

Processed row 7/1937
Sentence: 分享 了 顺和 牛肉 火锅 , 在 华阳 , 关于 牛 肉类 的 都 不错 , 建议 点牛肝 , 特色 牛肉 图 中 码满 辣椒面 的 那 [MASK] 怕 辣 的 , 肠胃 不 好 的 , 友情 建议 最好 不 要 尝试 哈 。
Target: 盘
Top1 prediction: 种 (logprob: -0.3557)
Top3 predictions: 种, 个, 是
Top3 log probabilities: -0.3557, -2.4102, -2.6503


Processing rows:   1%|          | 11/1937 [00:00<02:27, 13.08it/s]

...


Processed row 1929/1937
Sentence: 我 在 等 发 工资 了 会计 说 礼拜天 会 把 工资 打 在 我 的 卡上 我 等 啊 等 啊 心急 的 第一 [MASK] 工资
Target: 笔
Top1 prediction: 个 (logprob: -1.8085)
Top3 predictions: 个, 笔, 发
Top3 log probabilities: -1.8085, -1.9888, -2.1997

Processed row 1930/1937
Sentence: 我 爸 的 外甥女 和 我 在 一 [MASK] 公司 , 但 我们 不熟 , 别人 以为 我们 不 认识 。
Target: 家
Top1 prediction: 家 (logprob: -0.3461)
Top3 predictions: 家, 个, 间
Top3 log probabilities: -0.3461, -1.4209, -4.0671

Processed row 1931/1937
Sentence: 最 搞笑 的 事 , 有 人 大年 初 一 [MASK] 厕所 !
Target: 通
Top1 prediction: 上 (logprob: -0.1762)
Top3 predictions: 上, 去, 进
Top3 log probabilities: -0.1762, -2.3565, -4.3304

Processed row 1932/1937
Sentence: 看 了 一 天 纪录片 , 最 喜欢 一 [MASK] 讲 罗伯特 肯尼迪 遗孀 Ethel 的 , 导演 就 是 她 女儿 。
Target: 部
Top1 prediction: 部 (logprob: -1.0609)
Top3 predictions: 部, 个, 次
Top3 log probabilities: -1.0609, -1.1759, -3.0573


Processing rows: 100%|█████████▉| 1934/1937 [02:16<00:00, 12.75it/s]


Processed row 1933/1937
Sentence: 我 刚刚 在 爱 问 共 享 资料 上 传 了 资料 , 欢迎 大家 下载 分享 高 一 英语 完形 填空 练习 一 [MASK] 已 打印 doc 更 多
Target: 份
Top1 prediction: 篇 (logprob: -2.3919)
Top3 predictions: 篇, 、, 下
Top3 log probabilities: -2.3919, -2.6247, -2.9611

Processed row 1934/1937
Sentence: 住 进去 一 天 楼 歪 了 花 了 大 价钱 , 买 [MASK] 所谓 的 台州 地区 最 大 纯 海景 住宅 小 区 的 高层 住宅 , 新 房子 搬进 去 不到 一 天 , 楼 歪 了 , 你 是 什么 心情 ?
Target: 套
Top1 prediction: 了 (logprob: -0.0721)
Top3 predictions: 了, 的, 到
Top3 log probabilities: -0.0721, -4.1393, -4.2830

Processed row 1935/1937
Sentence: 英语 考试 结束 了 , 辛苦 一 凡 同学 了 , 又 一 [MASK] 拯救 了 大家 明天 的 物化 考试 , 但愿 我们 的 座位 一如既往 的 好 !
Target: 次
Top1 prediction: 次 (logprob: -0.0441)
Top3 predictions: 次, 个, 天
Top3 log probabilities: -0.0441, -4.1926, -5.9089


Processing rows: 100%|██████████| 1937/1937 [02:17<00:00, 14.13it/s]


Processed row 1936/1937
Sentence: 每 天 都 这 时候 才 有 一 [MASK] 睡意 , 唉 !
Target: 丝
Top1 prediction: 点 (logprob: -0.5778)
Top3 predictions: 点, 丝, 些
Top3 log probabilities: -0.5778, -1.1213, -2.7492

Processed row 1937/1937
Sentence: 吸引力 法则 果然 厉害 , 我 真的 买到 20 [MASK] 的 票 了 , 今天 我 继续 吸引 能 换 张 不 那么 慢 的 。
Target: 号
Top1 prediction: % (logprob: -1.0109)
Top3 predictions: %, 元, 张
Top3 log probabilities: -1.0109, -1.6916, -2.5376

Results saved to bert_mlm_result.csv


In [13]:
import pandas as pd

# 读取BERT预测结果
df = pd.read_csv('bert_mlm_result.csv')

# 计算Top-1准确率（预测的第一个词是否等于正确答案）
top1_correct = df['top1_prediction'] == df['target']
top1_accuracy = top1_correct.mean()

# 计算Top-3准确率（正确答案是否出现在top3预测中）
def is_target_in_top3(row):
    top3_predictions = row['top3_predictions'].split(", ")
    return row['target'] in top3_predictions

top3_correct = df.apply(is_target_in_top3, axis=1)
top3_accuracy = top3_correct.mean()

# 打印结果
print(f"Top-1 Accuracy: {top1_accuracy:.4f} ({top1_correct.sum()}/{len(df)})")
print(f"Top-3 Accuracy: {top3_accuracy:.4f} ({top3_correct.sum()}/{len(df)})")

# 可选：保存带有正确性标记的新CSV
df['top1_correct'] = top1_correct
df['top3_correct'] = top3_correct
df.to_csv('bert_mlm_result_with_accuracy.csv', index=False, encoding='utf-8-sig')

Top-1 Accuracy: 0.5426 (1051/1937)
Top-3 Accuracy: 0.7284 (1411/1937)
